# Random Forest for Network Intrusion Detection on UNSW-NB15
**CST9 Final Project**

Pipeline: load data → clean → preprocess → mutual-information feature selection → baseline Random Forest → validation experiments (feature count, tuning) → final test evaluation → save model.

**Task:** binary classification (0 = Normal, 1 = Attack).

## 1. Setup

In [ ]:
import os
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from functools import partial

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, roc_curve,
                             confusion_matrix, classification_report)

RANDOM_STATE = 42

## 2. Load Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/unsw-nb15'
file_a = pd.read_csv(f'{DATA_DIR}/UNSW_NB15_training-set.csv')
file_b = pd.read_csv(f'{DATA_DIR}/UNSW_NB15_testing-set.csv')

# The CSV names are swapped relative to the official partitions:
# the larger file (175,341 rows) is the training set, the smaller (82,332) the test set.
train_df, test_df = (file_a, file_b) if len(file_a) > len(file_b) else (file_b, file_a)

print("Training:", train_df.shape)
print("Testing: ", test_df.shape)

## 3. Data Inspection

In [ ]:
display(train_df.head())
train_df.info()

In [ ]:
print("Missing values  - train:", train_df.isnull().sum().sum(), "| test:", test_df.isnull().sum().sum())
print("Duplicate rows  - train:", train_df.duplicated().sum(), "| test:", test_df.duplicated().sum())

print("\nTraining class distribution:")
print(train_df['label'].value_counts())
print("\nTesting class distribution:")
print(test_df['label'].value_counts())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, df, name in zip(axes, [train_df, test_df], ['Training', 'Testing']):
    sns.countplot(data=df, x='label', ax=ax)
    ax.set_title(f"{name} Class Distribution")
    ax.set_xlabel("Traffic Class (0 = Normal, 1 = Attack)")
    ax.set_ylabel("Records")
plt.tight_layout()
plt.show()

## 4. Cleaning
`id` is only a row index and `attack_cat` is the multi-class version of the label (target leakage for a binary task), so both are dropped.

In [ ]:
drop_cols = ['id', 'attack_cat']
train_clean = train_df.drop(columns=drop_cols)
test_clean = test_df.drop(columns=drop_cols)

num_only = train_clean.select_dtypes(include=np.number)
print("Infinite values in train:", np.isinf(num_only.to_numpy()).sum())
print("Cleaned shapes:", train_clean.shape, test_clean.shape)

X_train = train_clean.drop(columns='label')
y_train = train_clean['label']
X_test = test_clean.drop(columns='label')
y_test = test_clean['label']

categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
numerical_cols = X_train.select_dtypes(include=np.number).columns.tolist()
print("Categorical:", categorical_cols)
print("Numerical features:", len(numerical_cols))

## 5. Preprocessing
One-hot encode categorical features and standardize numeric ones. The transformer is **fit on training data only** to avoid leakage.

In [ ]:
preprocessor = ColumnTransformer([
    ('categorical', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols),
    ('numerical', StandardScaler(), numerical_cols),
])

X_train_t = preprocessor.fit_transform(X_train)
X_test_t = preprocessor.transform(X_test)
feature_names = preprocessor.get_feature_names_out()

print("Transformed train:", X_train_t.shape)
print("Transformed test: ", X_test_t.shape)

## 6. Feature Selection (Mutual Information)
Top 30 features by mutual information with the label. One-hot columns are flagged as discrete. Fit on training data only.

In [ ]:
K = 30
n_cat = len(preprocessor.named_transformers_['categorical'].get_feature_names_out(categorical_cols))
discrete_mask = np.array([True] * n_cat + [False] * len(numerical_cols))

selector = SelectKBest(
    score_func=partial(mutual_info_classif, discrete_features=discrete_mask, random_state=RANDOM_STATE),
    k=K,
)
X_train_sel = selector.fit_transform(X_train_t, y_train)
X_test_sel = selector.transform(X_test_t)
selected_features = feature_names[selector.get_support()]

print("Selected train:", X_train_sel.shape, "| Selected test:", X_test_sel.shape)
print("\nSelected features:")
for i, f in enumerate(selected_features, 1):
    print(f"{i:2d}. {f}")

In [ ]:
mi_scores = (pd.DataFrame({'Feature': feature_names, 'MI Score': selector.scores_})
             .sort_values('MI Score', ascending=False))

plt.figure(figsize=(12, 8))
sns.barplot(data=mi_scores.head(20), x='MI Score', y='Feature', color='steelblue')
plt.title("Top 20 Features by Mutual Information")
plt.tight_layout()
plt.show()

## 7. Class Balance
The training set is about 68% attack. Instead of resampling (e.g., SMOTE), every Random Forest below uses `class_weight='balanced'`.

In [ ]:
print(y_train.value_counts())
print((y_train.value_counts(normalize=True) * 100).round(2))

plt.figure(figsize=(6, 4))
sns.countplot(x=y_train)
plt.title("Training Class Distribution")
plt.xlabel("Traffic Class (0 = Normal, 1 = Attack)")
plt.ylabel("Records")
plt.show()

## 8. Evaluation Helper

In [ ]:
def evaluate(model, X, y, name="Model"):
    pred = model.predict(X)
    prob = model.predict_proba(X)[:, 1]
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {
        'Model': name,
        'Accuracy': accuracy_score(y, pred),
        'Precision': precision_score(y, pred),
        'Recall': recall_score(y, pred),
        'F1-Score': f1_score(y, pred),
        'ROC-AUC': roc_auc_score(y, prob),
        'FPR': fp / (fp + tn),
        'TN': tn, 'FP': fp, 'FN': fn, 'TP': tp,
    }

def new_rf(**kwargs):
    params = dict(n_estimators=100, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)
    params.update(kwargs)
    return RandomForestClassifier(**params)

## 9. Baseline Models on the Test Set

In [ ]:
results = []
train_times = {}

# (a) All 194 features
t0 = time.time()
rf_baseline = new_rf().fit(X_train_t, y_train)
train_times['Baseline RF (194 features)'] = time.time() - t0
results.append(evaluate(rf_baseline, X_test_t, y_test, 'Baseline RF (194 features)'))

# (b) Top 30 features
t0 = time.time()
rf_selected = new_rf().fit(X_train_sel, y_train)
train_times['Feature-Selected RF (30 features)'] = time.time() - t0
results.append(evaluate(rf_selected, X_test_sel, y_test, 'Feature-Selected RF (30 features)'))

pd.DataFrame(results)[['Model', 'Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC', 'FPR']].round(4)

In [ ]:
print(classification_report(y_test, rf_selected.predict(X_test_sel), target_names=['Normal', 'Attack']))

cm = confusion_matrix(y_test, rf_selected.predict(X_test_sel), labels=[0, 1])
plt.figure(figsize=(6, 4.5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Normal', 'Attack'], yticklabels=['Normal', 'Attack'])
plt.title("Feature-Selected RF Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

In [ ]:
prob = rf_selected.predict_proba(X_test_sel)[:, 1]
fpr, tpr, _ = roc_curve(y_test, prob)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, linewidth=2, label=f"Random Forest (AUC = {roc_auc_score(y_test, prob):.4f})")
plt.plot([0, 1], [0, 1], '--', color='gray', label='Random classifier')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - UNSW-NB15 Test Set")
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.show()

In [ ]:
rf_importance = (pd.DataFrame({'Feature': selected_features,
                               'Importance': rf_selected.feature_importances_})
                 .sort_values('Importance', ascending=False))

plt.figure(figsize=(12, 8))
sns.barplot(data=rf_importance.head(20), x='Importance', y='Feature', color='steelblue')
plt.title("Top 20 Random Forest Feature Importances")
plt.tight_layout()
plt.show()

## 10. Validation Experiments
A stratified 80/20 split of the **training data** is used for all tuning decisions, so the official test set is only used for final reporting.

In [ ]:
X_dev, X_val, y_dev, y_val = train_test_split(
    X_train, y_train, test_size=0.20, random_state=RANDOM_STATE, stratify=y_train)

dev_preprocessor = clone(preprocessor)
X_dev_t = dev_preprocessor.fit_transform(X_dev)
X_val_t = dev_preprocessor.transform(X_val)

print("Dev:", X_dev_t.shape, "| Validation:", X_val_t.shape)
print("Dev class counts:", y_dev.value_counts().to_dict())
print("Val class counts:", y_val.value_counts().to_dict())

### 10.1 Feature count comparison

In [ ]:
dev_names = dev_preprocessor.get_feature_names_out()
dev_discrete = np.array([n.startswith("categorical__") for n in dev_names])

t0 = time.time()
mi_dev = mutual_info_classif(X_dev_t, y_dev, discrete_features=dev_discrete, random_state=RANDOM_STATE)
print(f"MI computed in {time.time() - t0:.1f}s")
ranked_idx = np.argsort(mi_dev)[::-1]

rows = []
for k in [10, 20, 30, 40]:
    idx = ranked_idx[:k]
    t0 = time.time()
    m = new_rf().fit(X_dev_t[:, idx], y_dev)
    elapsed = time.time() - t0
    r = evaluate(m, X_val_t[:, idx], y_val)
    rows.append({'Features': k, **{c: r[c] for c in ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC', 'FPR']},
                 'Train Time (s)': elapsed})

feature_count_df = pd.DataFrame(rows)
display(feature_count_df.round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for metric in ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']:
    axes[0].plot(feature_count_df['Features'], feature_count_df[metric], marker='o', label=metric)
axes[0].set_xticks(feature_count_df['Features'])
axes[0].set_ylim(0.9, 1.0)
axes[0].set_xlabel("Number of Selected Features")
axes[0].set_ylabel("Validation Score")
axes[0].set_title("Performance by Feature Count")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].bar(feature_count_df['Features'].astype(str), feature_count_df['FPR'] * 100)
axes[1].set_xlabel("Number of Selected Features")
axes[1].set_ylabel("False Positive Rate (%)")
axes[1].set_title("FPR by Feature Count")

plt.tight_layout()
plt.show()

**Choice of k:** 30 and 40 features tie on validation F1, so the smaller set (**k = 30**) is used for the final model.

### 10.2 Hyperparameter tuning
`RandomizedSearchCV` on the development split using the top-30 features (ranked from the development data only).

In [ ]:
idx30 = ranked_idx[:30]
X_dev_30 = X_dev_t[:, idx30]
X_val_30 = X_val_t[:, idx30]

param_distributions = {
    'n_estimators': [50, 100],
    'max_depth': [15, 25, None],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
}

search = RandomizedSearchCV(
    estimator=RandomForestClassifier(class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1),
    param_distributions=param_distributions,
    n_iter=4,
    scoring='f1',
    cv=StratifiedKFold(n_splits=2, shuffle=True, random_state=RANDOM_STATE),
    random_state=RANDOM_STATE,
    n_jobs=1,
    verbose=1,
)

t0 = time.time()
search.fit(X_dev_30, y_dev)
print(f"Search finished in {time.time() - t0:.1f}s")
print("Best parameters:", search.best_params_)
print(f"Best CV F1: {search.best_score_:.4f}")

In [ ]:
val_tuned = evaluate(search.best_estimator_, X_val_30, y_val, 'Tuned RF (validation)')
pd.DataFrame([val_tuned])[['Model', 'Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC', 'FPR']].round(4)

## 11. Final Model
Retrain with the tuned parameters on the **full training set** (30 selected features) and evaluate once on the official test set.

In [ ]:
best_params = search.best_params_
print("Using parameters:", best_params)

t0 = time.time()
rf_final = new_rf(**best_params).fit(X_train_sel, y_train)
train_times['Tuned RF (30 features)'] = time.time() - t0

final_result = evaluate(rf_final, X_test_sel, y_test, 'Tuned RF (30 features)')
results.append(final_result)

y_pred_final = rf_final.predict(X_test_sel)
print(classification_report(y_test, y_pred_final, target_names=['Normal', 'Attack']))
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred_final, labels=[0, 1]))

## 12. Model Comparison (Test Set)

In [ ]:
comparison = pd.DataFrame(results)
comparison['Training Time (s)'] = comparison['Model'].map(train_times)
comparison = comparison[['Model', 'Accuracy', 'Precision', 'Recall', 'F1-Score',
                         'ROC-AUC', 'FPR', 'Training Time (s)']]
display(comparison.round(4))

In [ ]:
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
short_names = ['Baseline (194)', 'Selected (30)', 'Tuned (30)']
x = np.arange(len(metrics))
w = 0.25

plt.figure(figsize=(12, 6))
for i, name in enumerate(short_names):
    plt.bar(x + (i - 1) * w, comparison.loc[i, metrics].astype(float) * 100, w, label=name)
plt.xticks(x, metrics)
plt.ylabel("Score (%)")
plt.ylim(0, 110)
plt.title("Random Forest Model Comparison on UNSW-NB15 Test Set")
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colors = ['steelblue', 'orange', 'seagreen']

bars = axes[0].bar(short_names, comparison['FPR'] * 100, color=colors)
axes[0].set_title("False Positive Rate")
axes[0].set_ylabel("FPR (%)")
axes[0].set_ylim(0, 35)
for b in bars:
    axes[0].text(b.get_x() + b.get_width() / 2, b.get_height() + 0.5, f"{b.get_height():.2f}%", ha='center')

bars = axes[1].bar(short_names, comparison['Training Time (s)'], color=colors)
axes[1].set_title("Training Time")
axes[1].set_ylabel("Seconds")
for b in bars:
    axes[1].text(b.get_x() + b.get_width() / 2, b.get_height() + 0.5, f"{b.get_height():.1f}s", ha='center')

plt.tight_layout()
plt.show()

## 13. Save and Verify the Model
Saves the classifier, preprocessor, selector, and input column order, then reloads them and confirms the predictions are identical on the full test set.

In [ ]:
model_dir = f'{DATA_DIR}/model'
os.makedirs(model_dir, exist_ok=True)

joblib.dump(rf_final, f'{model_dir}/random_forest_final.pkl')
joblib.dump(preprocessor, f'{model_dir}/preprocessor_final.pkl')
joblib.dump(selector, f'{model_dir}/feature_selector_final.pkl')
joblib.dump(X_train.columns.tolist(), f'{model_dir}/input_columns.pkl')

print(os.listdir(model_dir))

In [ ]:
loaded_model = joblib.load(f'{model_dir}/random_forest_final.pkl')
loaded_preprocessor = joblib.load(f'{model_dir}/preprocessor_final.pkl')
loaded_selector = joblib.load(f'{model_dir}/feature_selector_final.pkl')
loaded_columns = joblib.load(f'{model_dir}/input_columns.pkl')

X_raw = X_test[loaded_columns]
y_pred_loaded = loaded_model.predict(loaded_selector.transform(loaded_preprocessor.transform(X_raw)))

print(f"Accuracy: {accuracy_score(y_test, y_pred_loaded):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred_loaded):.4f}")
print("Predictions identical to in-memory model:", np.array_equal(y_pred_loaded, y_pred_final))

## 14. Export Files for the Streamlit App
Creates compressed model files (smaller for GitHub) plus everything the app's Overview, Try a Prediction, and Model Performance pages display. Copy the contents of the `app_export` folder into the `model/` folder of your GitHub repo.

In [ ]:
import json, sklearn

export_dir = f'{DATA_DIR}/app_export'
os.makedirs(export_dir, exist_ok=True)

# ---- compressed model components
joblib.dump(rf_final, f'{export_dir}/random_forest_final.pkl', compress=3)
joblib.dump(preprocessor, f'{export_dir}/preprocessor_final.pkl', compress=3)
joblib.dump(selector, f'{export_dir}/feature_selector_final.pkl', compress=3)

# ---- test-set predictions of the final model
prob_final = rf_final.predict_proba(X_test_sel)[:, 1]
fpr_c, tpr_c, _ = roc_curve(y_test, prob_final)
grid = np.linspace(0, 1, 201)

clean_name = lambda s: s.replace('numerical__', '').replace('categorical__', '')
imp = (pd.DataFrame({'Feature': [clean_name(f) for f in selected_features],
                     'Importance': rf_final.feature_importances_})
       .sort_values('Importance', ascending=False))

attack_counts = train_df.loc[train_df['attack_cat'] != 'Normal', 'attack_cat'].value_counts()

meta = {
    # form defaults and dropdown options
    'columns': X_train.columns.tolist(),
    'categorical_cols': categorical_cols,
    'defaults': {c: (X_train[c].mode()[0] if c in categorical_cols else float(X_train[c].median()))
                 for c in X_train.columns},
    'categories': {c: sorted(X_train[c].unique().tolist()) for c in categorical_cols},
    # overview page
    'overview': {
        'rows': int(len(train_df)),
        'attack_rate': float(y_train.mean()),
        'attack_types': int(attack_counts.shape[0]),
        'attack_counts': {k: int(v) for k, v in attack_counts.items()},
        'test_rows': int(len(test_df)),
    },
    # performance page
    'test_metrics': {k: float(final_result[k]) for k in ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC', 'FPR']},
    'confusion': [[int(final_result['TN']), int(final_result['FP'])],
                  [int(final_result['FN']), int(final_result['TP'])]],
    'roc': {'fpr': grid.tolist(), 'tpr': np.interp(grid, fpr_c, tpr_c).tolist()},
    'feature_importance': {'features': imp['Feature'].head(10).tolist(),
                           'importance': imp['Importance'].head(10).tolist()},
    'best_params': search.best_params_,
    'validation': {k: float(val_tuned[k]) for k in ['Accuracy', 'F1-Score', 'FPR']},
    'comparison': comparison.round(4).to_dict('records'),
}
with open(f'{export_dir}/app_meta.json', 'w') as f:
    json.dump(meta, f, default=lambda o: o.item() if hasattr(o, 'item') else str(o))

# ---- sample tables
train_df.sample(10, random_state=RANDOM_STATE).sort_index().to_csv(f'{export_dir}/sample_train.csv', index=False)

rng = np.random.RandomState(RANDOM_STATE)
pos = rng.choice(len(X_test), 10, replace=False)
sample_pred = X_test.iloc[pos][['proto', 'service', 'state', 'dur', 'sbytes', 'dbytes', 'sttl', 'ct_state_ttl']].copy()
sample_pred['actual'] = y_test.iloc[pos].map({0: 'Normal', 1: 'Attack'}).values
sample_pred['predicted'] = pd.Series(y_pred_final[pos]).map({0: 'Normal', 1: 'Attack'}).values
sample_pred['attack_probability'] = prob_final[pos].round(3)
sample_pred.to_csv(f'{export_dir}/sample_predictions.csv', index=False)

# 300 labelled raw records: used by the app's Randomize button and as a CSV to try the upload
pos2 = rng.choice(len(X_test), 300, replace=False)
X_test.iloc[pos2].assign(label=y_test.iloc[pos2].values).to_csv(f'{export_dir}/sample_input.csv', index=False)

print("scikit-learn version (pin this in requirements.txt):", sklearn.__version__)
for fn in sorted(os.listdir(export_dir)):
    print(f"{fn:35s} {os.path.getsize(os.path.join(export_dir, fn)) / 1e6:8.2f} MB")